#oportunidades de marketing restaurantes#

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_business = spark.table("workspace.yelp_ing.silver_business")
df_review = spark.table("workspace.yelp_ing.silver_review")

# ========== CÁLCULO DOS PARÂMETROS GLOBAIS DO RATING BAYESIANO ==========

# 1. Calcula a mediana de reviews por estabelecimento na plataforma
review_counts_per_business = df_review.groupBy("business_id").agg(
    F.count("review_id").alias("review_count")
)
median_reviews = review_counts_per_business.approxQuantile("review_count", [0.5], 0.01)[0]
print(f"Mediana de reviews da plataforma: {median_reviews}")

# 2. Calcula a média global de estrelas de todos os estabelecimentos
avg_global_stars = df_review.agg(F.avg("stars")).collect()[0][0]
print(f"Média global de estrelas: {avg_global_stars:.4f}")

# ========== CÁLCULO DO RATING_YELP POR ESTABELECIMENTO ==========

df_joined = df_business.join(df_review, df_business.business_id == df_review.business_id, "inner")

df_result = (
    df_joined.groupBy(df_business.business_id, df_business.name)
    .agg(
        F.avg(df_review.stars).alias("avg_stars"),
        F.count(df_review.review_id).alias("review_count"),
        F.sum(df_review.stars).alias("sum_stars")
    )
)

# Aplica a fórmula do Rating Bayesiano Ponderado:
# rating_yelp = (mediana_reviews * média_global + soma_estrelas) / (mediana_reviews + count_reviews)
df_result = df_result.withColumn(
    "rating_yelp",
    F.round(
        (F.lit(median_reviews) * F.lit(avg_global_stars) + F.col("sum_stars")) / 
        (F.lit(median_reviews) + F.col("review_count")),
        3
    )
)

# Ordena por rating_yelp decrescente
df_result = df_result.orderBy(F.col("rating_yelp").desc()).limit(200)

print("\n✓ Campo rating_yelp calculado com sucesso!")
print(f"Fórmula: (mediana={median_reviews} * média_global={avg_global_stars:.4f} + soma_estrelas) / (mediana + review_count)")
print("Rating arredondado para 3 casas decimais")

In [0]:
display(df_result)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_business = spark.table("workspace.yelp_ing.silver_business")
df_review = spark.table("workspace.yelp_ing.silver_review")

# ========== CÁLCULO DOS PARÂMETROS GLOBAIS DO RATING BAYESIANO ==========

# 1. Calcula a mediana de reviews por estabelecimento na plataforma
review_counts_per_business = df_review.groupBy("business_id").agg(
    F.count("review_id").alias("review_count")
)
median_reviews = review_counts_per_business.approxQuantile("review_count", [0.5], 0.01)[0]
print(f"Mediana de reviews da plataforma: {median_reviews}")

# 2. Calcula a média global de estrelas de todos os estabelecimentos
avg_global_stars = df_review.agg(F.avg("stars")).collect()[0][0]
print(f"Média global de estrelas: {avg_global_stars:.4f}")

# ========== CÁLCULO DO RATING_YELP POR ESTABELECIMENTO ==========

df_joined = df_business.join(df_review, df_business.business_id == df_review.business_id, "inner")

df_result = (
    df_joined.groupBy(df_business.business_id, df_business.name)
    .agg(
        F.avg(df_review.stars).alias("avg_stars"),
        F.count(df_review.review_id).alias("review_count"),
        F.sum(df_review.stars).alias("sum_stars")
    )
)

# Aplica a fórmula do Rating Bayesiano Ponderado:
# rating_yelp = (mediana_reviews * média_global + soma_estrelas) / (mediana_reviews + count_reviews)
df_result = df_result.withColumn(
    "rating_yelp",
    F.round(
        (F.lit(median_reviews) * F.lit(avg_global_stars) + F.col("sum_stars")) / 
        (F.lit(median_reviews) + F.col("review_count")),
        3
    )
)

# Carrega gold_business e faz join com rating_yelp
df_gold_business = spark.table("workspace.yelp_ing.gold_business")

df_gold_marketing = df_gold_business.join(
    df_result.select("business_id", "rating_yelp", "avg_stars", "review_count"),
    on="business_id",
    how="left"
)

# ========== SALVAR TABELA GOLD_MARKETING ==========
df_gold_marketing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.yelp_ing.gold_marketing")

print("\n✓ Tabela gold_marketing salva com sucesso em workspace.yelp_ing.gold_marketing!")
print(f"Total de registros: {df_gold_marketing.count()}")

In [0]:
display(df_gold_business)